# Chapter 14 — Deep Computer Vision Using CNNs
Companion notebook to `part2_deep_learning/14_cnn_computer_vision.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.0

# CNNs expect a channel dimension
X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test = X_test[..., np.newaxis]

## A typical CNN architecture

In [ ]:
cnn_model = keras.models.Sequential([
    keras.layers.Conv2D(64, 7, activation="relu", padding="same", input_shape=[28, 28, 1]),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation="softmax"),
])
cnn_model.summary()

In [ ]:
cnn_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
history = cnn_model.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))
print("Test accuracy:", cnn_model.evaluate(X_test, y_test)[1])
print("Total params:", cnn_model.count_params())

## Transfer learning with a pretrained model (MobileNetV2, lightweight)

In [ ]:
# Using MobileNetV2 here instead of Xception purely for a smaller download / faster demo.
# The recipe (freeze -> train top -> unfreeze -> fine-tune) is identical either way.
base_model = keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(96, 96, 3))

avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
output = keras.layers.Dense(5, activation="softmax")(avg)   # e.g. 5 classes of your own
transfer_model = keras.Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

transfer_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
transfer_model.summary()
print("(Fit this on your own image dataset resized to 96x96x3 -- frozen base first, then unfreeze top layers with a lower learning rate.)")

## ✍️ Practice
1. Compare the CNN's test accuracy/parameter count above against the plain Dense MLP from Chapter 10.
2. Why does a CNN need far fewer parameters than a Dense network for the same image?
3. Fine-tune the transfer-learning model on a real small image dataset (e.g. `tf_flowers` via `tensorflow_datasets`) — freeze first, then unfreeze the top few layers with a lower LR.
4. Debug: a segmentation model built on a heavily-pooled classification backbone produces blurry boundaries — what property is responsible, and what's the general fix?

In [ ]:
# Your code / notes here